# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

> NOTE: DO NOT RUN THESE CELLS IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY

In [ ]:
#!pip install -qU ragas==0.2.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.7/175.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.6/411.6 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.8/454.8 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/1

In [ ]:
#!pip install -qU langchain-community==0.3.14 langchain-openai==0.2.14 unstructured==0.16.12 langgraph==0.2.61 langchain-qdrant==0.2.0

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /home/et/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/et/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [2]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [4]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Loan Data use-case!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [ ]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "/"data
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [6]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

/home/et/dev/ai-boot-camp/AIE7/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/segmenter.py:66: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):
/home/et/dev/ai-boot-camp/AIE7/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/lang/arabic.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
/home/et/dev/ai-boot-camp/AIE7/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/lang/persian.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)


Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [7]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [8]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs[:20]:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 20, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [9]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node '63bc32'. Skipping!
Property 'summary' already exists in node 'b75ad1'. Skipping!
Property 'summary' already exists in node 'a3c927'. Skipping!
Property 'summary' already exists in node '359f2b'. Skipping!
Property 'summary' already exists in node '71d60c'. Skipping!
Property 'summary' already exists in node 'eba837'. Skipping!
Property 'summary' already exists in node 'e687a9'. Skipping!
Property 'summary' already exists in node 'b76517'. Skipping!
Property 'summary' already exists in node '8ed282'. Skipping!
Property 'summary' already exists in node 'ffe5b3'. Skipping!
Property 'summary' already exists in node '9ec4ba'. Skipping!
Property 'summary' already exists in node 'd59d68'. Skipping!
Property 'summary' already exists in node '5a4588'. Skipping!
Property 'summary' already exists in node 'aca10e'. Skipping!


Applying CustomNodeFilter:   0%|          | 0/6 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/43 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node '63bc32'. Skipping!
Property 'summary_embedding' already exists in node 'a3c927'. Skipping!
Property 'summary_embedding' already exists in node '5a4588'. Skipping!
Property 'summary_embedding' already exists in node '71d60c'. Skipping!
Property 'summary_embedding' already exists in node 'b76517'. Skipping!
Property 'summary_embedding' already exists in node 'aca10e'. Skipping!
Property 'summary_embedding' already exists in node 'ffe5b3'. Skipping!
Property 'summary_embedding' already exists in node '359f2b'. Skipping!
Property 'summary_embedding' already exists in node 'e687a9'. Skipping!
Property 'summary_embedding' already exists in node '9ec4ba'. Skipping!
Property 'summary_embedding' already exists in node 'd59d68'. Skipping!
Property 'summary_embedding' already exists in node 'b75ad1'. Skipping!
Property 'summary_embedding' already exists in node '8ed282'. Skipping!
Property 'summary_embedding' already exists in node 'eba837'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 40, relationships: 478)

We can save and load our knowledge graphs as follows.

In [10]:
kg.save("loan_data_kg.json")
loan_data_kg = KnowledgeGraph.load("loan_data_kg.json")
loan_data_kg

KnowledgeGraph(nodes: 40, relationships: 478)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [11]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=loan_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [13]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

#### Answer #1:

`SingleHopSpecificQuerySynthesizer`: generates simple and direct questions

`MultiHopAbstractQuerySynthesizer`: creates complex questions for reasoning

`MultiHopSpecificQuerySynthesizer`: this for tesing procedural knowledge by generating detailed procedural questions


Finally, we can use our `TestSetGenerator` to generate our testset!

In [14]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,Whaat is the role of the School Participaation...,"[Chapter 1 Academic Years, Academic Calendars,...",The context does not provide specific details ...,single_hop_specifc_query_synthesizer
1,What does 34 CFR 668.3(a) specify regarding ac...,[Regulatory Citations Academic year minimums: ...,34 CFR 668.3(a) pertains to the academic year ...,single_hop_specifc_query_synthesizer
2,"What information does Volume 8, Chapter 3 prov...",[Inclusion of Clinical Work in a Standard Term...,"Volume 8, Chapter 3 offers guidance on includi...",single_hop_specifc_query_synthesizer
3,What are the key features of Non-Term Characte...,[Non-Term Characteristics A program that measu...,A program that measures progress in clock hour...,single_hop_specifc_query_synthesizer
4,What is Appendix A about in relation to credit...,[both the credit or clock hours and the weeks ...,Appendix A provides examples related to the ef...,single_hop_specifc_query_synthesizer
5,How do nonstandard terms affect Title IV progr...,[<1-hop>\n\nInclusion of Clinical Work in a St...,Nonstandard terms are defined as terms that do...,multi_hop_abstract_query_synthesizer
6,How do payment periods and weeks of instructio...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",The context explains that each eligible progra...,multi_hop_abstract_query_synthesizer
7,How do timing and schedul constraints of clinc...,[<1-hop>\n\nInclusion of Clinical Work in a St...,The inclusion of clinical work in a standard t...,multi_hop_abstract_query_synthesizer
8,How do the definitions of academic years in Vo...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",Volume 2 explains that academic year requireme...,multi_hop_specific_query_synthesizer
9,Volume 8 and Volume 7 what do they say about c...,[<1-hop>\n\nInclusion of Clinical Work in a St...,"Volume 8, Chapter 3 gives guidance on clinical...",multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [15]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs[:20], testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node 'e7bf73'. Skipping!
Property 'summary' already exists in node '93a326'. Skipping!
Property 'summary' already exists in node '903966'. Skipping!
Property 'summary' already exists in node '918a9b'. Skipping!
Property 'summary' already exists in node '2ae439'. Skipping!
Property 'summary' already exists in node 'e95ca9'. Skipping!
Property 'summary' already exists in node '4e745d'. Skipping!
Property 'summary' already exists in node 'e0314d'. Skipping!
Property 'summary' already exists in node '79dcdf'. Skipping!
Property 'summary' already exists in node '0f7be6'. Skipping!
Property 'summary' already exists in node '649bf0'. Skipping!
Property 'summary' already exists in node 'f71b19'. Skipping!
Property 'summary' already exists in node 'b8eb46'. Skipping!
Property 'summary' already exists in node '717a9c'. Skipping!


...

Property 'summary_embedding' already exists in node 'f71b19'. Skipping!
Property 'summary_embedding' already exists in node 'e95ca9'. Skipping!
Property 'summary_embedding' already exists in node 'e7bf73'. Skipping!
Property 'summary_embedding' already exists in node '4e745d'. Skipping!
Property 'summary_embedding' already exists in node '903966'. Skipping!
Property 'summary_embedding' already exists in node '2ae439'. Skipping!
Property 'summary_embedding' already exists in node '649bf0'. Skipping!
Property 'summary_embedding' already exists in node '918a9b'. Skipping!
Property 'summary_embedding' already exists in node '93a326'. Skipping!
Property 'summary_embedding' already exists in node 'e0314d'. Skipping!
Property 'summary_embedding' already exists in node '0f7be6'. Skipping!
Property 'summary_embedding' already exists in node '717a9c'. Skipping!
Property 'summary_embedding' already exists in node '79dcdf'. Skipping!
Property 'summary_embedding' already exists in node 'b8eb46'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [16]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,What is the significance of Chapter 2 in the c...,"[Chapter 1 Academic Years, Academic Calendars,...",The provided context does not include specific...,single_hop_specifc_query_synthesizer
1,Could you please explain the significance of 3...,[Regulatory Citations Academic year minimums: ...,Regulatory citations indicate that 34 CFR 668....,single_hop_specifc_query_synthesizer
2,Can you explain the significance of Volume 8 i...,[Inclusion of Clinical Work in a Standard Term...,"In Volume 8, Chapter 3, additional guidance is...",single_hop_specifc_query_synthesizer
3,"So like, what is Federal Work-Study and how do...",[Non-Term Characteristics A program that measu...,The context states that the Federal Work-Study...,single_hop_specifc_query_synthesizer
4,"So, if a student in a subscription-based progr...",[<1-hop>\n\nboth the credit or clock hours and...,"In a subscription-based program, the first two...",multi_hop_abstract_query_synthesizer
5,How do the different academic years for variou...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",The disbursement of federal student aid is inf...,multi_hop_abstract_query_synthesizer
6,How do the reguations governng Title IV progra...,[<1-hop>\n\nboth the credit or clock hours and...,The regulations state that for clock-hour or n...,multi_hop_abstract_query_synthesizer
7,How does 34 CFR 668.3(a) ensure regulatory com...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",34 CFR 668.3(a) establishes the minimum academ...,multi_hop_abstract_query_synthesizer
8,where appendix A or B help disbursement rules ...,[<1-hop>\n\nDisbursement Timing in Subscriptio...,disbursement timing in subscription programs i...,multi_hop_specific_query_synthesizer
9,How do Chapters 2 and 3 collectively inform th...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",Chapter 2 details the requirements for definin...,multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [17]:
from langsmith import Client

client = Client()

dataset_name = "Loan Synthetic Data"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Loan Synthetic Data"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [18]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [19]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [20]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [21]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [22]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan RAG"
)

In [23]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [24]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

For our LLM, we will be using TogetherAI's endpoints as well!

We're going to be using Meta Llama 3.1 70B Instruct Turbo - a powerful model which should get us powerful results!

In [25]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [26]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [27]:
rag_chain.invoke({"question" : "What kinds of loans are available?"})

'The kinds of loans available include:\n\n- Direct Subsidized Loans (available only to undergraduate students)\n- Direct Unsubsidized Loans\n- Direct PLUS Loans or student Federal PLUS Loans\n- Subsidized and Unsubsidized Federal Stafford Loans (made under the FFEL Program before July 1, 2010)\n- Federal SLS Loans\n- Federal PLUS Loans (made under the FFEL Program before July 1, 2010)\n\nAdditionally, Direct Unsubsidized Loans and Direct PLUS Loans can be used to replace the Satisfactory Academic Progress Indicator (SAI) and cover unmet need. Graduate or professional students are eligible only for Direct Unsubsidized Loans and Direct PLUS Loans, but not for Direct Subsidized Loans.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [28]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [29]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

empathy_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "empathy": "Is this response empathetic? Does it make the user feel like they are being heard?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

- `qa_evaluator`: factual accuracy
- `labeled_helpfulness_evaluator`: how helpful a response is
- `empathy_evaluator`: customer service quality

## LangSmith Evaluation

In [30]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'frosty-liquid-77' at:
https://smith.langchain.com/o/750ddb9f-5f75-492e-84ec-bd1afa0d60ec/datasets/5d7b9769-af29-4bd7-80d9-c181189cabdc/compare?selectedSessions=86006b3b-d1d4-45c1-9c8e-af24ec488353




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,How do Volume 8 and Volume 7 relate to the tim...,Based on the context provided:\n\nVolume 8 dis...,None,Volume 8 discusses the timing of disbursements...,1,1,0,9.341262,f0acedab-0304-4fb9-84f6-a847b7a30a1d,912be783-546b-4972-b24f-ae1182fa5560
1,How do Volume 2 and Volume 7 relate to the tim...,I don't know.,None,Volume 2 discusses the requirements for paymen...,0,0,0,1.024430,884ec3b3-7e0e-4653-975f-376e37040ecb,e9cd642c-0246-40d8-a3f4-a5d33fab6916
2,How do Chapters 2 and 3 collectively inform th...,I don't know.,None,Chapter 2 details the requirements for definin...,0,0,0,1.253273,3f7336e1-2f81-4abc-a402-699f4e2819e0,56ee2b06-4c96-4aa6-ae9a-456ef99de615
3,where appendix A or B help disbursement rules ...,Appendix B helps with disbursement rules in su...,None,disbursement timing in subscription programs i...,1,1,0,3.397501,0b369cc8-00fd-46b7-a7b8-8f295307b211,06959ea7-6593-4527-8fef-3826c4471529
4,How does 34 CFR 668.3(a) ensure regulatory com...,"Based on the provided context, 34 CFR 668.3(a)...",None,34 CFR 668.3(a) establishes the minimum academ...,1,1,0,6.373382,5656520c-6adc-4fc1-8b1b-1ba176b8fc38,7e90be8c-0e8a-4135-ad2f-9c2dfc080df8
5,How do the reguations governng Title IV progra...,The regulations governing Title IV programs li...,None,The regulations state that for clock-hour or n...,1,1,0,9.068579,74a93d2f-0d2b-4e28-9097-fbf113dd1fb1,7b384492-30d0-4dee-8eee-baaad9062108
6,How do the different academic years for variou...,The different academic years for various progr...,None,The disbursement of federal student aid is inf...,1,1,0,7.064406,a607d4c9-e2ef-4bb5-9c9a-707357e0b4ff,c7d4050c-0900-4553-a6f6-14fc05a8e7f2
7,"So, if a student in a subscription-based progr...","Based on the provided context, here's how disb...",None,"In a subscription-based program, the first two...",1,1,0,8.126102,ec218ac1-624c-4878-b5b7-b66e99f044a3,86fb680a-9b2b-4f9c-81de-830378385167
8,"So like, what is Federal Work-Study and how do...",Federal Work-Study (FWS) is a form of need-bas...,None,The context states that the Federal Work-Study...,1,1,0,5.733404,c508ba60-b125-4d6b-9601-d4bd196e4f43,cc9cf567-2ab3-4fcf-9a61-60cdf70b94e9
9,Can you explain the significance of Volume 8 i...,"Based on the provided context, Volume 8 provid...",None,"In Volume 8, Chapter 3, additional guidance is...",1,1,0,3.411730,a39a867c-bf95-4527-9256-f170dc112b86,154c329e-88b8-4848-b92c-5bb36f26c924


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [31]:
EMPATHY_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

You must answer the question using empathy and kindness, and make sure the user feels heard.

Context: {context}
Question: {question}
"""

empathy_rag_prompt = ChatPromptTemplate.from_template(EMPATHY_RAG_PROMPT)

In [32]:
rag_documents = docs

In [33]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

#### Answer #2

Bigger chunks equals more data to grab context for, the overlap will give us the link between chunks so they become related on the LLM's "eyes".

In [34]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

#### Answer #3:

By using the large one it will be slower but more accurate.

In [35]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan Data for RAG"
)

In [36]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [37]:
empathy_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | empathy_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [38]:
empathy_rag_chain.invoke({"question" : "What kinds of loans are available?"})

"Thank you for your thoughtful question. Based on the context provided, it seems there are several types of loans available to support students and their families with educational expenses:\n\n1. **Direct Subsidized Loans** – These are need-based loans where the government pays the interest while the student is in school at least half-time. The loan amount is limited to the student’s financial need.\n\n2. **Direct Unsubsidized Loans** – These loans are available to students regardless of financial need. Interest accrues while the student is in school, but these loans can be combined with subsidized loans to cover more costs.\n\n3. **Direct PLUS Loans** – These loans are for parents of dependent students or for graduate and professional students themselves. There is no fixed loan limit, but the loan cannot exceed the student's cost of attendance minus other financial aid. Parents must meet eligibility requirements to take out a Direct PLUS Loan.\n\n4. Additionally, if parents are unable

Finally, we can evaluate the new chain on the same test set!

In [39]:
evaluate(
    empathy_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "empathy_rag_chain"},
)

View the evaluation results for experiment: 'worthwhile-ink-40' at:
https://smith.langchain.com/o/750ddb9f-5f75-492e-84ec-bd1afa0d60ec/datasets/5d7b9769-af29-4bd7-80d9-c181189cabdc/compare?selectedSessions=fb68e9fc-4808-49e3-a3bf-b12c1a237585




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,How do Volume 8 and Volume 7 relate to the tim...,Thank you for your thoughtful question. Based ...,None,Volume 8 discusses the timing of disbursements...,0,0,1,7.801068,f0acedab-0304-4fb9-84f6-a847b7a30a1d,118b0e61-93d9-4b05-b678-85a10e745d63
1,How do Volume 2 and Volume 7 relate to the tim...,Thank you for your thoughtful question. Based ...,None,Volume 2 discusses the requirements for paymen...,0,0,1,6.490344,884ec3b3-7e0e-4653-975f-376e37040ecb,edca09bc-8672-4278-ac8e-c7b6164aa333
2,How do Chapters 2 and 3 collectively inform th...,Thank you for your thoughtful question. Based ...,None,Chapter 2 details the requirements for definin...,0,0,1,5.343945,3f7336e1-2f81-4abc-a402-699f4e2819e0,a43e3e28-6b34-475f-b7cc-7ae3787b2a25
3,where appendix A or B help disbursement rules ...,Thank you for your thoughtful question. From t...,None,disbursement timing in subscription programs i...,1,1,1,2.497844,0b369cc8-00fd-46b7-a7b8-8f295307b211,df2d6664-d33d-4ec8-9ea1-4819f13fa623
4,How does 34 CFR 668.3(a) ensure regulatory com...,Thank you for your thoughtful question. I unde...,None,34 CFR 668.3(a) establishes the minimum academ...,1,1,1,3.886906,5656520c-6adc-4fc1-8b1b-1ba176b8fc38,6060fe1c-b276-435f-8935-3e76dc02463e
5,How do the reguations governng Title IV progra...,Thank you for your thoughtful question. Based ...,None,The regulations state that for clock-hour or n...,1,0,1,5.529548,74a93d2f-0d2b-4e28-9097-fbf113dd1fb1,f9b52165-24d6-4653-ace4-fa6f123b6b9d
6,How do the different academic years for variou...,Thank you for your thoughtful question. Unders...,None,The disbursement of federal student aid is inf...,1,1,1,6.575623,a607d4c9-e2ef-4bb5-9c9a-707357e0b4ff,42eefbdf-e477-4b5c-b748-faaa5034dafc
7,"So, if a student in a subscription-based progr...",Thank you for your thoughtful question — it’s ...,None,"In a subscription-based program, the first two...",1,1,1,6.269750,ec218ac1-624c-4878-b5b7-b66e99f044a3,f2183adc-7524-4b73-922a-6c2e181f34b7
8,"So like, what is Federal Work-Study and how do...",Thank you for your question—it's completely un...,None,The context states that the Federal Work-Study...,1,1,1,4.637235,c508ba60-b125-4d6b-9601-d4bd196e4f43,1a0a95e2-d054-414e-8ee6-17436965a001
9,Can you explain the significance of Volume 8 i...,Thank you for your thoughtful question. Based ...,None,"In Volume 8, Chapter 3, additional guidance is...",1,1,1,5.937151,a39a867c-bf95-4527-9256-f170dc112b86,de3fd95f-db13-40c2-934b-47110ee63316


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.

There was an improvement on the empathy metric which was 100% on the addition of the empathy RAG prompt.

About the delay I dont believe it has something to do with that prompt.

![ragas_diff](data/ragas_diff.png)